# Sequence Model Testbed

This notebook is a reusable roundtrip testbed for the current sequence model.

It does four things for the selected Guitar Pro input:

1. Converts the score to Motif mapped JSON and a `.motif` source package with `motif-cli`
2. Rebuilds the training-time token surface and samples the current checkpoint
3. Accepts only generated bars whose onset layout and full-bar duration coverage exactly match the source last-bar template
4. Writes accepted bars back to JSON and exports a new `.gp` file using `--source-score` so Motif can reattach app-specific extension data

The export gate is intentionally strict right now. If the model does not emit structurally complete bars, the notebook will keep the analysis artifacts and skip GP export instead of writing an incoherent score.

The current helper is tuned for the simpler single-track `temp/SequenceTestFiles` inputs. Multi-track files with simultaneous onset groups can still hit a known template-alignment limitation, and the notebook will report that explicitly instead of exporting a broken score.


In [1]:
from pathlib import Path

REPO_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
SEQUENCE_TEST_ROOT = REPO_ROOT / "temp" / "SequenceTestFiles"
AVAILABLE_SEQUENCE_TEST_FILES = tuple(
    path.relative_to(REPO_ROOT) for path in sorted(SEQUENCE_TEST_ROOT.glob("*.gp"))
)

INPUT_SCORE_PATH = Path("temp/SequenceTestFiles/ChordTest1.gp")
REQUESTED_COMPLETE_BARS = 1

# Optional knobs
TOKEN_OVERSAMPLING_FACTOR = 8
FAIL_IF_INSUFFICIENT_COMPLETE_BARS = True
SHOW_GENERATED_BAR_LIMIT = 12
RUN_SEQUENCE_TEST_SWEEP = False

print("Available sequence test files:")
for path in AVAILABLE_SEQUENCE_TEST_FILES:
    print(f" - {path}")
print(f"\nSelected input: {INPUT_SCORE_PATH}")
print(f"Requested complete bars: {REQUESTED_COMPLETE_BARS}")

Available sequence test files:
 - temp/SequenceTestFiles/ChordTest1.gp
 - temp/SequenceTestFiles/ChordTest2.gp
 - temp/SequenceTestFiles/ChordTest3.gp
 - temp/SequenceTestFiles/ChordTest4.gp
 - temp/SequenceTestFiles/ChordTest5.gp
 - temp/SequenceTestFiles/MelodicTest.gp

Selected input: temp/SequenceTestFiles/ChordTest1.gp
Requested complete bars: 1


In [2]:
from motifml.evaluation.sequence_testbed import (
    formatted_bar_summaries,
    result_summary_lines,
    run_sequence_testbed,
)


def pitch_label_from_note(note: dict) -> str:
    pitch = note["pitch"]
    accidental = pitch.get("accidental") or ""
    accidental = accidental.replace("b", "B")
    return f"{pitch['step']}{accidental}{pitch['octave']}"


def describe_measure(measure: dict) -> list[str]:
    descriptions: list[str] = []
    for beat in measure["voices"][0]["beats"]:
        if not beat.get("notes"):
            continue
        offset = beat["offset"]
        descriptions.append(
            f"{offset['numerator']}/{offset['denominator']}: "
            f"{[pitch_label_from_note(note) for note in beat['notes']]}"
        )
    return descriptions

In [3]:
result = None
run_error = None

try:
    result = run_sequence_testbed(
        repo_root=REPO_ROOT,
        input_score_path=INPUT_SCORE_PATH,
        requested_complete_bars=REQUESTED_COMPLETE_BARS,
        token_oversampling_factor=TOKEN_OVERSAMPLING_FACTOR,
        fail_if_insufficient_complete_bars=FAIL_IF_INSUFFICIENT_COMPLETE_BARS,
    )
except Exception as exc:  # Notebook-facing guard so unsupported inputs fail readably.
    run_error = exc
    print(f"Sequence testbed run failed: {type(exc).__name__}: {exc}")
else:
    for line in result_summary_lines(result):
        print(line)

    print("\nArtifacts:")
    print(f"- Source mapped JSON: {result.paths.source_json_path}")
    print(f"- Source Motif package: {result.paths.source_motif_path}")
    print(f"- Continuation JSON: {result.paths.output_json_path}")
    print(f"- Continuation GP: {result.paths.output_gp_path}")
    print(f"- Roundtrip verify JSON: {result.paths.verify_json_path}")
    print(f"- Summary JSON: {result.paths.summary_path}")

Input score: /home/mjrasicci/Repos/MotifML/temp/SequenceTestFiles/ChordTest1.gp
Requested complete bars: 1
Accepted complete bars: 0
Export performed: False
Export reason: insufficient_complete_bars_for_strict_export
Sampling: 256 generated tokens on cuda from checkpoint epoch-0004.pt
Template onset counts by track: Acoustic Guitar=1
Template bar durations by track: Acoustic Guitar=11531520 ticks

Artifacts:
- Source mapped JSON: /home/mjrasicci/Repos/MotifML/temp/roundtrip/sequence_testbed/temp__SequenceTestFiles__ChordTest1/ChordTest1.mapped.json
- Source Motif package: /home/mjrasicci/Repos/MotifML/temp/roundtrip/sequence_testbed/temp__SequenceTestFiles__ChordTest1/ChordTest1.source.motif
- Continuation JSON: /home/mjrasicci/Repos/MotifML/temp/roundtrip/sequence_testbed/temp__SequenceTestFiles__ChordTest1/ChordTest1.1bars.continued.json
- Continuation GP: /home/mjrasicci/Repos/MotifML/temp/roundtrip/sequence_testbed/temp__SequenceTestFiles__ChordTest1/ChordTest1.1bars.continued.gp
-

In [4]:
if result is None:
    print(
        "No template summary is available because the selected input could not be prepared for strict roundtrip export."
    )
else:
    print("Source last-bar template:")
    for template in result.template_tracks:
        print(
            f"- {template.track_name}: {template.onset_count} onsets, "
            f"{template.bar_duration_ticks} ticks"
        )
        for onset_template in template.onset_templates:
            offset = onset_template.beat_template["offset"]
            print(
                f"  {offset['numerator']}/{offset['denominator']} "
                f"({onset_template.offset_ticks} ticks): "
                f"{list(onset_template.json_pitch_labels)}"
            )

    print("\nGenerated bar summaries:")
    bar_blocks = formatted_bar_summaries(result)
    for block in bar_blocks[:SHOW_GENERATED_BAR_LIMIT]:
        for line in block:
            print(line)
        print()

    if len(bar_blocks) > SHOW_GENERATED_BAR_LIMIT:
        print(
            f"... truncated {len(bar_blocks) - SHOW_GENERATED_BAR_LIMIT} additional generated bars."
        )

Source last-bar template:
- Acoustic Guitar: 1 onsets, 11531520 ticks
  0/1 (0 ticks): ['A5', 'D5', 'C5', 'F4']

Generated bar summaries:
Generated bar 1: leading_time_shift_ticks=11531520, onsets=1, notes=0, complete=False, exported=False
  Rejection reason: generated onset note count does not match the source last-bar template.
  Raw onset offsets: [0]
  Raw note labels: []

Generated bar 2: leading_time_shift_ticks=0, onsets=1, notes=0, complete=False, exported=False
  Rejection reason: generated onset note count does not match the source last-bar template.
  Raw onset offsets: [0]
  Raw note labels: []

Generated bar 3: leading_time_shift_ticks=0, onsets=1, notes=0, complete=False, exported=False
  Rejection reason: generated onset note count does not match the source last-bar template.
  Raw onset offsets: [0]
  Raw note labels: []

Generated bar 4: leading_time_shift_ticks=0, onsets=1, notes=0, complete=False, exported=False
  Rejection reason: generated onset note count does not

In [5]:
if result is None:
    print(
        "No GP export check was run because the selected input failed before strict export evaluation."
    )
elif not result.export_performed or result.verified_score is None:
    print(
        "No GP export was written because the generated continuation did not pass the strict structural gate."
    )
else:
    source_bar_count = len(result.source_score["timelineBars"])
    verified_bar_count = len(result.verified_score["timelineBars"])

    print(f"Original bar count: {source_bar_count}")
    print(f"Verified output bar count: {verified_bar_count}")
    print(f"Exported GP: {result.paths.output_gp_path}")
    print(f"Roundtrip verification JSON: {result.paths.verify_json_path}")
    print("\nAppended measures by track:")
    for track in result.verified_score["tracks"]:
        print(f"- {track['name']}")
        appended_measures = track["staves"][0]["measures"][source_bar_count:]
        for measure in appended_measures:
            print(f"  Measure {measure['index'] + 1}:")
            for description in describe_measure(measure):
                print(f"    {description}")

No GP export was written because the generated continuation did not pass the strict structural gate.


In [6]:
if RUN_SEQUENCE_TEST_SWEEP:
    print("Sequence test sweep:")
    for relative_path in AVAILABLE_SEQUENCE_TEST_FILES:
        try:
            sweep_result = run_sequence_testbed(
                repo_root=REPO_ROOT,
                input_score_path=relative_path,
                requested_complete_bars=REQUESTED_COMPLETE_BARS,
                token_oversampling_factor=TOKEN_OVERSAMPLING_FACTOR,
                fail_if_insufficient_complete_bars=FAIL_IF_INSUFFICIENT_COMPLETE_BARS,
            )
        except Exception as exc:
            print(f"\n{relative_path}")
            print(f"  error={type(exc).__name__}: {exc}")
            continue
        first_failure = next(
            (
                summary.rejection_reason or summary.parse_error
                for summary in sweep_result.generated_bar_summaries
                if not summary.is_structurally_complete
            ),
            None,
        )
        print(f"\n{relative_path}")
        print(f"  accepted_complete_bars={sweep_result.accepted_complete_bar_count}")
        print(f"  export_reason={sweep_result.export_reason}")
        print(f"  first_failure={first_failure}")
else:
    print(
        "Set RUN_SEQUENCE_TEST_SWEEP = True to evaluate every file in temp/SequenceTestFiles."
    )

Set RUN_SEQUENCE_TEST_SWEEP = True to evaluate every file in temp/SequenceTestFiles.
